In [1]:
#!pip install netCDF4
from netCDF4 import Dataset
import math
import numpy as np
import time
from scipy.stats import gamma
import os
from datetime import timedelta, datetime

Data Process Script - Standard Element 2 - Tuomas Haapala

This file takes the gridded precipitation data stored in raw_data folder, calculates the SPI time series for each grid cell and saves the SPI map in a csv along with key values used for later analysis.

User-specified inputs define the accumulation period used for SPI calculation and the discretisation step for determining grid resolution.

This function is for creating a summer mask to allow study of the summer / growing season.


In [2]:
def summer_flag(all_time_data):
    """This function is for creating a summer mask to allow study of the summer / growing season."""
    summer = []
    length = len(all_time_data)
    a = [timedelta(x) + datetime(1970,1,1) for x in all_time_data]
    for i in range(length):
        kk = a[i].month
        if kk < 9 and kk > 4:
            summer.append(1)
        else:
            summer.append(0)
    return a, summer

In [13]:
discretisationstep = int(input("Please enter the size of the grid cell used ('1' for one square kilometer, '10' for 10 km * 10 km, '100' for 100 km * 100 km.)\n"))
accumulation_period_input = int(input("Please enter the accumulation period ('1' for SPI-1, '3' for SPI-3, '6' for SPI-6, '12' for SPI-12)\n"))
accumulation_periods_dictionary = {1:30, 3:90, 6:180, 12:360} 
accumulation_period = accumulation_periods_dictionary[accumulation_period_input] #Set the amount of days used for each accumulation period.


raw_data_list = os.listdir(path='raw_data')
if raw_data_list[0] == ".ipynb_checkpoints":
    del raw_data_list[0]
raw_data_list.sort()
file_dictionary = {}
for i in range(len(raw_data_list)):
    file_dictionary[i] = 'raw_data/' + raw_data_list[i]
file = Dataset(file_dictionary.get(0), mode='r')
files = [0, len(raw_data_list)-1]
analysis_length = files[1] - files[0] + 1
lons = file.variables['Lon'][:] #longitudes and latitudes to determine the spatial extent of the data
lats = file.variables['Lat'][:]
prtemp = file.variables['RRday'][:] 
file.close()

length_of_data = 0
for x in range(files[0], files[1]+1):
    file = Dataset(file_dictionary.get(x), mode='r')
    length_of_data += len(file.variables['RRday'][:])
"""Grid size in 1kmx1km"""
imax = len(lats)
jmax = len(lons)
"""Modify grid size according to discretisation step"""
imaxpick = int(imax / discretisationstep)  # Lat index
jmaxpick = int(jmax / discretisationstep)  # Lon index

spi_map = np.zeros((imaxpick,jmaxpick,length_of_data))
"""SPI constants for calculation (Lloyd-Hughes)"""
c_0 = 2.515517
c_1 = 0.802853
c_2 = 0.010328
d_1 = 1.432788
d_2 = 0.189269
d_3 = 0.001308

temp_name = 'Time'
precipitationdatas = []
progress_milestones = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]



Please enter the size of the grid cell used ('1' for one square kilometer, '10' for 10 km * 10 km, '100' for 100 km * 100 km.)
 10
Please enter the accumulation period ('1' for SPI-1, '3' for SPI-3, '6' for SPI-6, '12' for SPI-12)
 1


This loop calculates the SPI time series for each cell in the gridded data, with the discretisation step used to "skip over" cells to save on resolution and computing time.

In [14]:
timer_start = time.time()
for i in range(imaxpick):
    for j in range(jmaxpick):
        all_precipitation_data = []
        all_time_data = []
        long_SPI_list = []
        icurrent = i*discretisationstep
        jcurrent = j*discretisationstep
        """Check whether the current cell has data via comparison against the fill value of the data array."""
        if prtemp[0,icurrent,jcurrent] != -3.4e+38: # check for non-mask values
            for n in range(files[0], files[1]+1): #Cell identified. Go through data year by year.
                file = Dataset(file_dictionary.get(n), mode='r') #lue tiedosto.
                precipitation_data = file.variables['RRday'][:,icurrent,jcurrent]
                time_data = file.variables[temp_name][:]
                all_precipitation_data = np.append(all_precipitation_data, precipitation_data)
                all_time_data = np.append(all_time_data, time_data)
            total_sum_precipitations = np.zeros(len(all_precipitation_data))
            precipitationdatas.append(all_precipitation_data)
            average_sum_precipitation = np.mean(all_precipitation_data)*accumulation_period
            for m in range(len(all_precipitation_data)):
                if m < (accumulation_period - 1):
                    total_sum_precipitations[m] = average_sum_precipitation
                else:
                    total_sum_precipitations[m] = sum(all_precipitation_data[m-(accumulation_period - 1):m+1])
            """Input dataset created. Calculation of additional variables for SPI calculation"""
            count = len(total_sum_precipitations)
            spi_series_average = sum(total_sum_precipitations) / count
            spi_series_ln_average = math.log(spi_series_average)
            spi_series_ln_precipitations = []
            """Occurrence of zero values in the dataset."""
            count_zero = 0
            for o in range(len(total_sum_precipitations)):
                if total_sum_precipitations[o] == 0:
                    count_zero +=1
                    spi_series_ln_precipitations.append(0)
                else:
                    spi_series_ln_precipitations.append(math.log(total_sum_precipitations[o]))
            spi_series_ln_precipitations_sum = sum(spi_series_ln_precipitations)
            variable_a = spi_series_ln_average - (spi_series_ln_precipitations_sum/count)
            variable_alpha = (1/(4*variable_a))*(1+math.sqrt(1+(4/3)*variable_a))
            variable_beta = spi_series_average / variable_alpha
            """Cumulative distribution function required."""
            """cdf(x,a,loc=o,scale=1) t. scipystats"""
            #a = shape = alpha, loc = shift = location, scale = scale = beta
            """G(x)"""
            gamma_distro_function = gamma.cdf(total_sum_precipitations, variable_alpha, loc=0, scale=variable_beta)
            variable_q = count_zero / count
            """H(x) = q+(1-q)*G(x)"""
            cumulative_probability_function = variable_q + (1-variable_q)*gamma_distro_function
            h = 0
            SPI_list = []
            for h in range(len(cumulative_probability_function)):
                if cumulative_probability_function[h] > 0.5:
                    """IF 0.5 < H(x) < 1"""
                    variable_t = math.sqrt(math.log(1/((1-(cumulative_probability_function[h]))**2)))
                    SPI_value = variable_t - ((c_0 + c_1*variable_t + c_2*variable_t**2)/ \
                                              (1+d_1*variable_t+d_2*variable_t**2+d_3*variable_t**3))
                    SPI_list.append(SPI_value)
                else:
                    """IF 0 < H(x) <= 0.5"""                            
                    variable_t = math.sqrt(math.log(1/(cumulative_probability_function[h])**2))
                    SPI_value = -1* (variable_t - ((c_0 + c_1*variable_t + c_2*variable_t**2)/ \
                                                   (1+d_1*variable_t+d_2*variable_t**2+d_3*variable_t**3)))
                    SPI_list.append(SPI_value)
                long_SPI_list.append(SPI_value)
            spi_map[i,j] = long_SPI_list
            summer, summer_mask = summer_flag(all_time_data)

    progress = round(i/imaxpick,1)
    if progress in progress_milestones:
        print(int(progress*100), "%")
        progress_milestones.remove(progress)

timer_end = time.time()
duration = timer_end - timer_start
print(round(duration,1), "seconds elapsed.")

10 %
20 %
30 %
40 %
50 %
60 %
70 %
80 %
90 %
100 %
238.1 seconds elapsed.


In [11]:
"""Saving the SPI map and key indicators into analysis_folder as csv files for later reading"""
def create_analysis_folder():
    analysis_folder = os.getcwd() + "/analysis_folder" 
    try:
        os.makedirs(analysis_folder)
    except OSError as iex:
        print(f"Creation of directories failed! {iex}")
    return analysis_folder

analysis_folder = create_analysis_folder()

spi_map_ravelled = np.ravel(spi_map) #turn 3dmap to 1d for saving
np.savetxt(analysis_folder + "/spi_map_ravelled.csv", spi_map_ravelled, delimiter=",")
np.savetxt(analysis_folder + '/key_discstep_datalengths.csv', [discretisationstep, len(raw_data_list), length_of_data], delimiter=",")

summer_array = np.array(summer)
np.savetxt(analysis_folder + '/summer.csv', summer_array, delimiter = ",", fmt='%s')
np.savetxt(analysis_folder + '/summer_mask.csv', summer_mask, delimiter = ",")            


Creation of directories failed! [Errno 17] File exists: '/home/jovyan/WaterDig/ProjectFiles/analysis_folder'
